In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/2_preprocess/step3_area.csv", low_memory=False)
df["BIRTH_YMD"]  = pd.to_datetime(df["BIRTH_YMD"], errors="coerce")
df["ABATT_DATE"] = pd.to_datetime(df["ABATT_DATE"], errors="coerce")
df["JUDGE_DATE"] = pd.to_datetime(df["JUDGE_DATE"], errors="coerce")

print(f"로드 완료: {df.shape}")

lineage_cols = ["KPN_NO", "FATHER_CATTLE_NO", "MOTHER_ANIMAL_NO",
                "F_GMOTHER_ANIMAL_NO", "F_GFATHER_CATTLE_NO",
                "M_GMOTHER_ANIMAL_NO", "M_GFATHER_CATTLE_NO"]

for col in lineage_cols:
    print(f"\n=== {col} ===")
    print(f"결측 수: {df[col].isnull().sum():,}")
    print(f"고유값 수: {df[col].nunique():,}")
    print(f"상위 5개:\n{df[col].value_counts().head()}")

로드 완료: (2408699, 44)

=== KPN_NO ===
결측 수: 926,223
고유값 수: 1,367
상위 5개:
KPN_NO
L4pAahin8b605Ls6aXRPxNxksh7YJ/Oar7DbRr/VO6E=    82251
J/gD3xkm4LmGvPc8DqbEyrRIJPce2NDJ+JVM7xipcMc=    19564
xgWzvSdK/AYzp6NP6chU9gFMIWLjjQeHgekG2ExQmRg=    19254
qj3Z5fgag0LyAq7NPyeUJ0g/lUxWr7hKCHm0LOBB1O8=    18147
POTMbaWlSKbtzC2KC3FZe+sse6e6aLFYHnRd6WJQFbc=    18067
Name: count, dtype: int64

=== FATHER_CATTLE_NO ===
결측 수: 922,653
고유값 수: 1,342
상위 5개:
FATHER_CATTLE_NO
kluWj1LiM8I6nYWfDenO7q4tJySB2AVV8z9cMqweuXA=    82617
1LIDC2B6FG6MDnaUBU6Ke9At1pUZJaI3bjuRYh1GBvc=    19564
3hVMMUbR0msN0QQmXwxpOw4+uINFm7Bx3wrS+k0UoyE=    19254
Mdcdf56OkMwp/PSmp/j5Wj/uBOegsjWSQDqJof7Ppis=    18147
GqFSeVc3dFg4kjnMc7ti0uJFRgmNUKNdi+K4bmpWBs8=    18067
Name: count, dtype: int64

=== MOTHER_ANIMAL_NO ===
결측 수: 922,653
고유값 수: 1,051,304
상위 5개:
MOTHER_ANIMAL_NO
kluWj1LiM8I6nYWfDenO7q4tJySB2AVV8z9cMqweuXA=    82748
gQagjD++POKUI4kyvXKUoA==                         3862
hQ/Pi6O/ySexul8suK+U/3Hv/xzdvKEkwvEU3jVcv6Q=      104
oPwO2BrUvY

In [2]:
suspicious = "gQagjD++POKUI4kyvXKUoA=="

# 이 해시가 CATTLE_NO로 존재하는지 확인
print(f"CATTLE_NO에 존재: {(df['CATTLE_NO'] == suspicious).sum():,}")

# 각 컬럼별 등장 횟수
for col in lineage_cols:
    cnt = (df[col] == suspicious).sum()
    if cnt > 0:
        print(f"{col}: {cnt:,}개")

CATTLE_NO에 존재: 0
FATHER_CATTLE_NO: 3,620개
MOTHER_ANIMAL_NO: 3,862개
F_GMOTHER_ANIMAL_NO: 86,120개
F_GFATHER_CATTLE_NO: 85,845개
M_GMOTHER_ANIMAL_NO: 91,745개
M_GFATHER_CATTLE_NO: 91,423개


In [3]:
raw_lineage = pd.read_csv("../../data/raw/hanwoo_lineage.csv", low_memory=False)
print(f"원본 lineage CATTLE_NO에 존재: {(raw_lineage['CATTLE_NO'] == suspicious).sum():,}")
print(f"원본 lineage KPN_NO에 존재: {(raw_lineage['KPN_NO'] == suspicious).sum():,}")

# 이 값을 가진 행의 다른 컬럼 확인
sample = raw_lineage[raw_lineage['FATHER_CATTLE_NO'] == suspicious].head(3)
print(f"\n원본에서 FATHER_CATTLE_NO가 해당 해시인 샘플:")
print(sample)

원본 lineage CATTLE_NO에 존재: 0
원본 lineage KPN_NO에 존재: 0

원본에서 FATHER_CATTLE_NO가 해당 해시인 샘플:
                                        CATTLE_NO KPN_NO  \
308  w4eaG1ve96FB9RDTIzJsHau9HD/GHrKV+t38QQAODVE=    -99   
448  YCDskTvuARxF7lk/HSBbWcJ9KvtHudhuJs0AF4m3wu8=    -99   
633  ph4/v/LBqeW0r82Cq5vx1fp012Vd/2Fd1nACKc5IRmE=    -99   

             FATHER_CATTLE_NO          MOTHER_ANIMAL_NO  \
308  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   
448  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   
633  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   

          F_GMOTHER_ANIMAL_NO       F_GFATHER_CATTLE_NO  \
308  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   
448  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   
633  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==   

          M_GMOTHER_ANIMAL_NO       M_GFATHER_CATTLE_NO  
308  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==  
448  gQagjD++POKUI4kyvXKUoA==  gQagjD++POKUI4kyvXKUoA==  
633  gQagjD++POKUI4kyvX

In [4]:
# lineage_df의 CATTLE_NO에 이 해시가 있는지 확인
print(f"lineage CATTLE_NO에 존재: {(raw_lineage['CATTLE_NO'] == suspicious).sum():,}")

# 이 해시를 가진 행들의 KPN_NO 패턴 확인
has_suspicious = raw_lineage[raw_lineage['FATHER_CATTLE_NO'] == suspicious]
print(f"\n해당 해시를 FATHER로 가진 행 수: {len(has_suspicious):,}")
print(f"KPN_NO 분포:")
print(has_suspicious['KPN_NO'].value_counts().head())

lineage CATTLE_NO에 존재: 0

해당 해시를 FATHER로 가진 행 수: 4,553
KPN_NO 분포:
KPN_NO
-99                                             4480
S6pm+lyhHvpWNz5CDZrJdHCNDpSXhkMN114PZS3frDg=      44
S1X+r6R1wCOhhJushVh2clf3wTQbHmcUCOWy10FiI2w=       8
L4pAahin8b605Ls6aXRPxNxksh7YJ/Oar7DbRr/VO6E=       7
SAkMVgL6dlZ2D1nF/rsJ7F0Uitypcso38MBbUhnKrwk=       4
Name: count, dtype: int64


In [5]:
df.to_csv("../../data/processed/2_preprocess/step4_lineage.csv",
          index=False, encoding="utf-8-sig")
print(f"저장 완료: {df.shape}")

저장 완료: (2408699, 44)
